In [ ]:
# ==========================================
# 1. SETUP & IMPORTS
# ==========================================
import os, time, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from paddleocr import TextRecognition

# Riproducibilità
random.seed(42); np.random.seed(42); torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device in uso: {DEVICE}")

# Vocabolario CCPD
PROVINCES = ['皖','沪','津','渝','冀','晋','蒙','辽','吉','黑','苏','浙','京',
             '闽','赣','鲁','豫','鄂','湘','粤','桂','琼','川','贵','云','藏',
             '陕','甘','青','宁','新','警','学','O'] 
ALPHABETS = ['A','B','C','D','E','F','G','H','J','K','L','M','N',
             'P','Q','R','S','T','U','V','W','X','Y','Z','O'] 
ADS = ['A','B','C','D','E','F','G','H','J','K','L','M','N',
       'P','Q','R','S','T','U','V','W','X','Y','Z',
       '0','1','2','3','4','5','6','7','8','9','O']

VOCAB = list(dict.fromkeys(PROVINCES[:-1] + [c for c in (ALPHABETS+ADS) if c != 'O'] + ['O']))
char2idx = {c: i for i, c in enumerate(VOCAB)}
idx2char = {i: c for c, i in char2idx.items()}

NUM_CLASSES = len(VOCAB)
BLANK_IDX = NUM_CLASSES # Usato per la CTCLoss in LPRNet

In [ ]:
# ==========================================
# 2. DATASET & STRATIFIED K-FOLD
# ==========================================

class PlateCropDataset(Dataset):
    def __init__(self, df, augment=False, out_w=94, out_h=24):
        self.df = df
        self.augment = augment
        self.out_w, self.out_h = out_w, out_h

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row["path"])
        if img is None:
            img = np.zeros((self.out_h, self.out_w, 3), dtype=np.uint8)
        else:
            img = cv2.resize(img, (self.out_w, self.out_h))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
        # Normalizzazione [-1, 1] per LPRNet
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 127.5 - 1.0
        
        plate_str = row["plate"]
        target = [char2idx[c] for c in plate_str if c in char2idx]
        return img_tensor, torch.tensor(target, dtype=torch.long), len(target), plate_str

def collate(batch):
    imgs, targets, target_lens, plates = zip(*batch)
    imgs = torch.stack(imgs)
    targets = torch.cat(targets)
    target_lens = torch.tensor(target_lens, dtype=torch.long)
    return imgs, targets, target_lens, plates

# Funzione per il K-Fold stratificato
def make_stratified_folds(df, n_folds, seed=42):
    rng = random.Random(seed)
    by_subset = {}
    for i, r in df.iterrows():
        by_subset.setdefault(r["subset"], []).append(i)
    fold_of = {}
    for subset, ids in by_subset.items():
        ids = list(ids); rng.shuffle(ids)
        for j, idx in enumerate(ids):
            fold_of[idx] = j % n_folds
    df = df.copy()
    df["fold"] = df.index.map(fold_of)
    return df


In [ ]:
# ==========================================
# CARICAMENTO DEL DATAFRAME ORIGINALE
# ==========================================

# 1. Definiamo i percorsi esatti come nel tuo notebook originale
WORK_ROOT = Path("crnn_plate")
LABELS_CSV = WORK_ROOT / "labels.csv"

# 2. Carichiamo il dataset (assicurandoci che il file esista)
if LABELS_CSV.exists():
    df_crops = pd.read_csv(LABELS_CSV)
    print(f"Dataset caricato con successo: {len(df_crops)} immagini trovate.")
else:
    raise FileNotFoundError(f"Errore: il file {LABELS_CSV} non esiste! Esegui prima la generazione dei crop.")

# Aggiungiamo la colonna della lunghezza della targa (utile per alcune analisi)
df_crops["plate_len"] = df_crops["plate"].str.len()

# 3. Separazione in POOL (Train + Validation per il K-Fold) e TEST (Frozen per la fine)
pool_df = df_crops[df_crops["split"]=="pool"].reset_index(drop=True)
test_df = df_crops[df_crops["split"]=="test"].reset_index(drop=True)

# 4. Applichiamo la funzione per creare i 4 fold stratificati
N_FOLDS = 4
pool_df = make_stratified_folds(pool_df, n_folds=N_FOLDS, seed=0)

# 5. Stampa di controllo 
print(f"Pool size (Train+Val): {len(pool_df)} | Test size (Frozen): {len(test_df)}")
print("\nDistribuzione per Fold e Subset:")
print(pool_df.groupby(["subset", "fold"]).size().unstack(fill_value=0))

In [ ]:
# ==========================================
# 3. MODELLI: LPRNet & 7-Head CNN
# ==========================================

class small_basic_block(nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        ch_mid = ch_out // 4
        self.block = nn.Sequential(
            nn.Conv2d(ch_in,  ch_mid, kernel_size=1), nn.ReLU(),
            nn.Conv2d(ch_mid, ch_mid, kernel_size=(3, 1), padding=(1, 0)), nn.ReLU(),
            nn.Conv2d(ch_mid, ch_mid, kernel_size=(1, 3), padding=(0, 1)), nn.ReLU(),
            nn.Conv2d(ch_mid, ch_out, kernel_size=1),
        )
    def forward(self, x): return self.block(x)

class LPRNet(nn.Module):
    def __init__(self, class_num=NUM_CLASSES, dropout_rate=0.5):
        super().__init__()
        self.class_num = class_num
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(3, 1),
            small_basic_block(64, 128), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(3, (1, 2)),
            small_basic_block(128, 256), nn.BatchNorm2d(256), nn.ReLU(),
            small_basic_block(256, 256), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(3, (1, 2)),
            nn.Dropout(dropout_rate),
            nn.Conv2d(256, 256, (1, 4), 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Conv2d(256, class_num, (13, 1)), nn.BatchNorm2d(class_num), nn.ReLU(),
        )
        # Init: bias the blank logit negative — keeps the all-blank attractor weaker.
        with torch.no_grad():
            self.container.bias.zero_()
            self.container.bias[BLANK_IDX] = -2.0
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None and m is not self.container:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        # Collect global-context features from layers 2, 6, 13, 22
        keep_layers = {2, 6, 13, 22}
        feats = []
        for i, layer in enumerate(self.backbone):
            x = layer(x)
            if i in keep_layers:
                feats.append(x)
        # Power-normalize each captured feature: f / mean(f**2).
        # Then resize each to match the final feature map's H×W, concat along channels.
        final = feats[-1]
        H_out, W_out = final.shape[2], final.shape[3]
        normed = []
        for f in feats:
            f_pow = f.pow(2)
            f_mean = f_pow.mean()
            f_norm = f.div(f_mean.clamp(min=1e-8))
            if f.shape[2:] != (H_out, W_out):
                f_norm = F.adaptive_avg_pool2d(f_norm, (H_out, W_out))
            normed.append(f_norm)
        cat = torch.cat(normed, dim=1)                    # B x (64+128+256+class_num) x H_out x W_out
        logits = self.container(cat)                       # B x class_num x H_out x W_out
        # Average along the height axis → sequence of length W_out
        logits = logits.mean(dim=2)                        # B x class_num x W_out
        logits = logits.permute(2, 0, 1).contiguous()      # T x B x V  (ready for CTCLoss)
        return logits
    
CROP_H, CROP_W = LPRNET_CROP_H, LPRNET_CROP_W
print(f"Crops will be resized to {CROP_H}×{CROP_W} (H×W) for LPRNet")

class SevenHeadCNN(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, n_slots=7):
        super().__init__()
        def blk(ci, co):
            return nn.Sequential(nn.Conv2d(ci, co, 3, 1, 1), nn.BatchNorm2d(co),
                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.backbone = nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128), nn.AdaptiveAvgPool2d(1))
        self.heads = nn.ModuleList([nn.Linear(128, n_classes) for _ in range(n_slots)])

    def forward(self, x):
        f = self.backbone(x).flatten(1)
        return torch.stack([h(f) for h in self.heads], dim=1) # B x 7 x V per CrossEntropy

In [ ]:
# TEST LPRNET
def greedy_ctc_decode(logits_TBV):
    '''logits: T x B x V (raw or log-softmax — argmax is the same). Returns list[str].'''
    # argmax along V → T x B; then we go per-batch and collapse
    idx = logits_TBV.argmax(dim=2).cpu().numpy()      # T x B
    T, B = idx.shape
    out = []
    for b in range(B):
        seq = []
        prev = -1
        for t in range(T):
            c = int(idx[t, b])
            if c != prev and c != BLANK_IDX:
                seq.append(IDX2CHAR[c])
            prev = c
        out.append(''.join(seq))
    return out

def edit_distance(a: str, b: str) -> int:
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    dp = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = prev if ca == cb else 1 + min(prev, dp[j], dp[j-1])
            prev = cur
    return dp[-1]

def evaluate(model, loader, device):
    model.eval()
    full_correct = 0; total = 0; ed_sum = 0
    pos_correct = np.zeros(7, dtype=np.int64); pos_total = np.zeros(7, dtype=np.int64)
    with torch.no_grad():
        for xb, _, _, plates in loader:
            xb = xb.to(device)
            logits = model(xb)                              # T x B x V
            preds = greedy_ctc_decode(logits)
            for pred, truth in zip(preds, plates):
                total += 1
                ed_sum += edit_distance(pred, truth)
                if pred == truth: full_correct += 1
                if len(truth) == 7:
                    for i in range(7):
                        pos_total[i] += 1
                        if i < len(pred) and pred[i] == truth[i]:
                            pos_correct[i] += 1
    return {
        "full_plate_acc": full_correct / max(total, 1),
        "mean_edit_dist": ed_sum / max(total, 1),
        "per_position_acc": (pos_correct / np.maximum(pos_total, 1)).tolist(),
        "n": total,
    }

# tiny sanity test of decoder
_logits = torch.full((10, 1, N_CLASSES), -10.0)
# craft a sequence: [BLANK, BLANK, '皖', '皖', BLANK, 'A', BLANK, '1', BLANK, BLANK]
seq = [BLANK_IDX, BLANK_IDX, CHAR2IDX['皖'], CHAR2IDX['皖'], BLANK_IDX,
       CHAR2IDX['A'], BLANK_IDX, CHAR2IDX['1'], BLANK_IDX, BLANK_IDX]
for t, c in enumerate(seq):
    _logits[t, 0, c] = 10.0
print("decoder test → expect '皖A1':", greedy_ctc_decode(_logits))


In [ ]:
# ==========================================
# 5A. DATA MINING: K-FOLD TRAINING (LPRNet)
# ==========================================
import time
import torch.nn.functional as F


EPOCHS_CV = 30
BATCH_SIZE = 128
LR = 1e-3
PATIENCE = 20

cv_results_lprnet = []
lprnet_histories = []

print(f"=== Inizio K-Fold Cross Validation per LPRNet ({N_FOLDS} folds) ===")

for k in range(N_FOLDS):
    print(f"\n{'='*15} FOLD {k} {'='*15}")
    train_df = pool_df[pool_df["fold"] != k].reset_index(drop=True)
    val_df   = pool_df[pool_df["fold"] == k].reset_index(drop=True)
    
    train_loader = DataLoader(PlateCropDataset(train_df, augment=True), batch_size=BATCH_SIZE, 
                              shuffle=True, num_workers=0, collate_fn=collate, pin_memory=(DEVICE=="cuda"))
    val_loader   = DataLoader(PlateCropDataset(val_df, augment=False), batch_size=BATCH_SIZE, 
                              shuffle=False, num_workers=0, collate_fn=collate, pin_memory=(DEVICE=="cuda"))

    model_lpr = LPRNet(class_num=NUM_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(model_lpr.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS_CV)
    ctc = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)

    best_edit = float("inf")
    best_acc = 0.0
    best_state = None
    stale = 0
    history = []
    
    # --- TRAINING LOOP LPRNET ---
    for ep in range(1, EPOCHS_CV + 1):
        model_lpr.train()
        loss_sum = 0.0
        n = 0
        t0 = time.time()
        
        for xb, targets, target_lens, _ in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE)
            target_lens = target_lens.to(DEVICE)
            
            logits = model_lpr(xb)                                      # T x B x V
            log_probs = F.log_softmax(logits, dim=2)
            T_, B, _ = log_probs.shape
            input_lens = torch.full((B,), T_, dtype=torch.long, device=DEVICE)
            
            loss = ctc(log_probs, targets, input_lens, target_lens)
            optim.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model_lpr.parameters(), 5.0)
            optim.step()
            
            loss_sum += loss.item() * B
            n += B
            
        sched.step()
        train_loss = loss_sum / max(n, 1)

        # --- VALIDATION LOOP (EARLY STOPPING) ---
        # NOTA: Assicurati che la tua funzione 'evaluate' sia definita e supporti LPRNet!
        val_metrics = evaluate(model_lpr, val_loader, DEVICE)
        val_acc = val_metrics["full_plate_acc"]
        val_edit = val_metrics["mean_edit_dist"]
        
        history.append({"epoch": ep, "train_loss": train_loss, "val_full_acc": val_acc, "val_edit": val_edit})
        
        dt = time.time() - t0
        print(f"  ep {ep:3d} | loss {train_loss:.4f} | val full-plate {val_acc:.4f} | edit {val_edit:.3f} | {dt:.1f}s")
        
        if val_edit < best_edit:
            best_edit = val_edit
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model_lpr.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"  early stop @ epoch {ep} (best edit={best_edit:.3f}, full-plate={best_acc:.4f})")
                break

    cv_results_lprnet.append(best_acc)
    lprnet_histories.append(history)
    print(f">>> LPRNet Fold {k} Completato. Best Acc: {best_acc:.4f}")

print("\n" + "="*50)
print(f"LPRNet - Media Accuratezza K-Fold: {np.mean(cv_results_lprnet):.4f} ± {np.std(cv_results_lprnet):.4f}")
print("="*50)

In [ ]:
# ==========================================
# 5B. DATA MINING: K-FOLD TRAINING (7-Head CNN)
# ==========================================

cv_results_cnn = []
cnn_histories = []

print(f"=== Inizio K-Fold Cross Validation per 7-Head CNN ({N_FOLDS} folds) ===")

for k in range(N_FOLDS):
    print(f"\n{'='*15} FOLD {k} {'='*15}")
    train_df = pool_df[pool_df["fold"] != k].reset_index(drop=True)
    val_df   = pool_df[pool_df["fold"] == k].reset_index(drop=True)
    
    train_loader = DataLoader(PlateCropDataset(train_df, augment=True), batch_size=BATCH_SIZE, 
                              shuffle=True, num_workers=0, collate_fn=collate, pin_memory=(DEVICE=="cuda"))
    val_loader   = DataLoader(PlateCropDataset(val_df, augment=False), batch_size=BATCH_SIZE, 
                              shuffle=False, num_workers=0, collate_fn=collate, pin_memory=(DEVICE=="cuda"))

    model_cnn = SevenHeadCNN(n_classes=NUM_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(model_cnn.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS_CV)
    ce_loss = nn.CrossEntropyLoss()

    best_acc = 0.0
    best_state = None
    stale = 0
    history = []
    
    # --- TRAINING LOOP 7-HEAD CNN ---
    for ep in range(1, EPOCHS_CV + 1):
        model_cnn.train()
        loss_sum = 0.0
        n = 0
        t0 = time.time()
        
        for xb, targets, _, _ in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE).long()
            
            # Forward pass
            logits = model_cnn(xb) # B x 7 x V
            
            # Calcolo Loss per le 7 teste
            loss = 0
            for i in range(7):
                loss += ce_loss(logits[:, i, :], targets[:, i])
                
            optim.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model_cnn.parameters(), 5.0)
            optim.step()
            
            loss_sum += loss.item() * xb.size(0)
            n += xb.size(0)
            
        sched.step()
        train_loss = loss_sum / max(n, 1)

        # --- VALIDATION LOOP (EARLY STOPPING SULLA ACCURACY) ---
        model_cnn.eval()
        correct_plates = 0
        total_plates = 0
        
        with torch.no_grad():
            for xb, targets, _, _ in val_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE).long()
                
                logits = model_cnn(xb)
                preds = logits.argmax(dim=2) # Shape: (B, 7)
                
                # Full-plate match: tutte e 7 le predizioni devono essere corrette
                correct = (preds == targets).all(dim=1).sum().item()
                correct_plates += correct
                total_plates += xb.size(0)
                
        val_acc = correct_plates / total_plates
        history.append({"epoch": ep, "train_loss": train_loss, "val_full_acc": val_acc})
        
        dt = time.time() - t0
        print(f"  ep {ep:3d} | loss {train_loss:.4f} | val full-plate {val_acc:.4f} | {dt:.1f}s")
        
        # Early Stopping basato sulla Full-Plate Accuracy
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model_cnn.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"  early stop @ epoch {ep} (best full-plate={best_acc:.4f})")
                break

    cv_results_cnn.append(best_acc)
    cnn_histories.append(history)
    print(f">>> 7-Head CNN Fold {k} Completato. Best Acc: {best_acc:.4f}")

print("\n" + "="*50)
print(f"7-Head CNN - Media Accuratezza K-Fold: {np.mean(cv_results_cnn):.4f} ± {np.std(cv_results_cnn):.4f}")
print("="*50)